# 03 - Temporal Data Splitting

## Objective
This notebook implements a chronological data splitting strategy for the H&M Personalized Fashion Recommendations dataset.

Unlike random splitting, temporal splitting preserves the real-world order of transactions. This helps evaluate recommendation models using historical customer behaviour to predict future purchases.

## Experimental Setup

The dataset will be divided into three chronological periods:

- **Training:** 2018-09-20 to 2020-07-31
- **Validation:** 2020-08-01 to 2020-08-31
- **Testing:** 2020-09-01 to 2020-09-22

The training set will be used to build recommendation models, the validation set to tune model parameters, and the test set for final evaluation.

## Leakage Prevention

Only information available before each prediction period should be used to construct model features. The test set must remain untouched during model development and tuning.

In [4]:
import duckdb
import pandas as pd

# Connect to the existing H&M research database
con = duckdb.connect("../data/processed/hm_recommendations.duckdb")

print("DuckDB connection established successfully!")

DuckDB connection established successfully!


In [5]:
date_summary = con.execute("""
    SELECT
        MIN(t_dat) AS first_transaction,
        MAX(t_dat) AS last_transaction,
        COUNT(*) AS total_transactions
    FROM transactions
""").df()

date_summary

,first_transaction,last_transaction,total_transactions
0,2018-09-20,2020-09-22,31788324


In [6]:
split_summary = con.execute("""
    SELECT
        CASE
            WHEN t_dat <= DATE '2020-07-31' THEN 'Training'
            WHEN t_dat <= DATE '2020-08-31' THEN 'Validation'
            ELSE 'Testing'
        END AS dataset,
        MIN(t_dat) AS start_date,
        MAX(t_dat) AS end_date,
        COUNT(*) AS transaction_count,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(DISTINCT article_id) AS unique_articles
    FROM transactions
    GROUP BY dataset
    ORDER BY
        CASE dataset
            WHEN 'Training' THEN 1
            WHEN 'Validation' THEN 2
            WHEN 'Testing' THEN 3
        END
""").df()

split_summary

,dataset,start_date,end_date,transaction_count,unique_customers,unique_articles
0,Training,2018-09-20,2020-07-31,29752863,1321769,98416
1,Validation,2020-08-01,2020-08-31,1237192,254163,30162
2,Testing,2020-09-01,2020-09-22,798269,189510,26252


In [7]:
overlap_summary = con.execute("""
WITH
train AS (
    SELECT DISTINCT customer_id, article_id
    FROM transactions
    WHERE t_dat <= DATE '2020-07-31'
),
validation AS (
    SELECT DISTINCT customer_id, article_id
    FROM transactions
    WHERE t_dat BETWEEN DATE '2020-08-01' AND DATE '2020-08-31'
),
testing AS (
    SELECT DISTINCT customer_id, article_id
    FROM transactions
    WHERE t_dat BETWEEN DATE '2020-09-01' AND DATE '2020-09-22'
)
SELECT
    (SELECT COUNT(*) FROM train) AS train_customer_article_pairs,
    (SELECT COUNT(*) FROM validation) AS validation_customer_article_pairs,
    (SELECT COUNT(*) FROM testing) AS test_customer_article_pairs,

    (SELECT COUNT(*) FROM train t
     INNER JOIN validation v USING (customer_id)) AS train_validation_customer_overlap,

    (SELECT COUNT(*) FROM train t
     INNER JOIN testing s USING (customer_id)) AS train_test_customer_overlap,

    (SELECT COUNT(*) FROM validation v
     INNER JOIN testing s USING (customer_id)) AS validation_test_customer_overlap
""").df()

overlap_summary

,train_customer_article_pairs,validation_customer_article_pairs,test_customer_article_pairs,train_validation_customer_overlap,train_test_customer_overlap,validation_test_customer_overlap
0,25550610,1096492,705504,65306128,41650015,2223006


In [8]:
customer_overlap = con.execute("""
WITH
train_customers AS (
    SELECT DISTINCT customer_id
    FROM transactions
    WHERE t_dat <= DATE '2020-07-31'
),
validation_customers AS (
    SELECT DISTINCT customer_id
    FROM transactions
    WHERE t_dat BETWEEN DATE '2020-08-01' AND DATE '2020-08-31'
),
test_customers AS (
    SELECT DISTINCT customer_id
    FROM transactions
    WHERE t_dat BETWEEN DATE '2020-09-01' AND DATE '2020-09-22'
)
SELECT
    (SELECT COUNT(*) FROM train_customers) AS train_customers,
    (SELECT COUNT(*) FROM validation_customers) AS validation_customers,
    (SELECT COUNT(*) FROM test_customers) AS test_customers,

    (SELECT COUNT(*)
     FROM train_customers t
     INNER JOIN validation_customers v USING (customer_id)
    ) AS train_validation_overlap,

    (SELECT COUNT(*)
     FROM train_customers t
     INNER JOIN test_customers s USING (customer_id)
    ) AS train_test_overlap,

    (SELECT COUNT(*)
     FROM validation_customers v
     INNER JOIN test_customers s USING (customer_id)
    ) AS validation_test_overlap
""").df()

customer_overlap

,train_customers,validation_customers,test_customers,train_validation_overlap,train_test_overlap,validation_test_overlap
0,1321769,254163,189510,231189,169700,79875


In [9]:
article_overlap = con.execute("""
WITH
train_articles AS (
    SELECT DISTINCT article_id
    FROM transactions
    WHERE t_dat <= DATE '2020-07-31'
),
validation_articles AS (
    SELECT DISTINCT article_id
    FROM transactions
    WHERE t_dat BETWEEN DATE '2020-08-01' AND DATE '2020-08-31'
),
test_articles AS (
    SELECT DISTINCT article_id
    FROM transactions
    WHERE t_dat BETWEEN DATE '2020-09-01' AND DATE '2020-09-22'
)
SELECT
    (SELECT COUNT(*) FROM train_articles) AS train_articles,
    (SELECT COUNT(*) FROM validation_articles) AS validation_articles,
    (SELECT COUNT(*) FROM test_articles) AS test_articles,

    (SELECT COUNT(*)
     FROM train_articles t
     INNER JOIN validation_articles v USING (article_id)
    ) AS train_validation_overlap,

    (SELECT COUNT(*)
     FROM train_articles t
     INNER JOIN test_articles s USING (article_id)
    ) AS train_test_overlap,

    (SELECT COUNT(*)
     FROM validation_articles v
     INNER JOIN test_articles s USING (article_id)
    ) AS validation_test_overlap
""").df()

article_overlap

,train_articles,validation_articles,test_articles,train_validation_overlap,train_test_overlap,validation_test_overlap
0,98416,30162,26252,26726,20441,21544


In [10]:
cold_start_summary = con.execute("""
WITH
train_customers AS (
    SELECT DISTINCT customer_id
    FROM transactions
    WHERE t_dat <= DATE '2020-07-31'
),
train_articles AS (
    SELECT DISTINCT article_id
    FROM transactions
    WHERE t_dat <= DATE '2020-07-31'
),
future_data AS (
    SELECT
        CASE
            WHEN t_dat BETWEEN DATE '2020-08-01'
                 AND DATE '2020-08-31' THEN 'Validation'
            ELSE 'Testing'
        END AS dataset,
        customer_id,
        article_id
    FROM transactions
    WHERE t_dat BETWEEN DATE '2020-08-01'
                    AND DATE '2020-09-22'
),
unique_future AS (
    SELECT DISTINCT dataset, customer_id, article_id
    FROM future_data
)
SELECT
    dataset,
    COUNT(DISTINCT customer_id) AS total_customers,
    COUNT(DISTINCT CASE
        WHEN customer_id NOT IN (SELECT customer_id FROM train_customers)
        THEN customer_id END) AS new_customers,
    COUNT(DISTINCT article_id) AS total_articles,
    COUNT(DISTINCT CASE
        WHEN article_id NOT IN (SELECT article_id FROM train_articles)
        THEN article_id END) AS new_articles
FROM unique_future
GROUP BY dataset
ORDER BY
    CASE dataset
        WHEN 'Validation' THEN 1
        ELSE 2
    END
""").df()

cold_start_summary

,dataset,total_customers,new_customers,total_articles,new_articles
0,Validation,254163,22974,30162,3436
1,Testing,189510,19810,26252,5811


## Cold-Start Analysis Findings

The temporal split revealed cold-start challenges in both customers and articles.

### Validation Period
- 22,974 customers (9.04%) were not observed in training.
- 3,436 articles (11.39%) were not observed in training.

### Testing Period
- 19,810 customers (10.45%) were not observed in training.
- 5,811 articles (22.14%) were not observed in training.

### Research Implications

The test period contains a higher proportion of previously unseen articles than the validation period. This presents a significant challenge for collaborative filtering approaches that depend on historical user-item interactions.

Content-based recommendation methods may help address this limitation by using product metadata independently of historical purchase interactions.

These findings support evaluating hybrid recommendation approaches alongside collaborative filtering and content-based methods.

Note: "New" refers to entities not observed in the training transaction period, rather than necessarily newly registered customers or newly launched products.